# EDA — Green Turning Point (GTP)
## Análisis Exploratorio de Datos · Justificación del Pipeline ETL

**Universidad Europea · Big Data I · 3º Grado GIAMAD**  
**Autores:** Juan Manuel Palencia · Pablo Mata · Pablo Sánchez · María Paula Aguirre

---

Este notebook documenta el análisis exploratorio de las **12 fuentes de datos** del proyecto,
justificando cada decisión de limpieza y transformación aplicada en el pipeline ETL.

| # | Fuente | Variable principal | Script |
|---|--------|--------------------|--------|
| 1 | Sentinel-2 (GEE) | NDVI mensual | `Sentinel-2_extract.py` |
| 2 | Sentinel-5P NO₂ (GEE) | Dióxido de nitrógeno | `Sentinel-5p_extract.py` |
| 3 | HRL Copernicus | Impermeabilización suelo | `hrl_to_csv.py` |
| 4 | ERA5-Land (GEE) | Temperatura / Precipitación | `era5_extract.py` |
| 5 | Sentinel-5P UVAI (GEE) | Índice aerosoles UV | `s5p_aerosol_extract.py` |
| 6 | ESA WorldCover (GEE) | Cobertura del suelo | `urban_atlas_extract.py` |
| 7 | EDGAR v8 | Emisiones CO₂ nacionales | `edgar_co2_extract.py` |
| 8 | Yahoo Finance | Empresas verdes cotizadas | `YFinance_extract.py` |
| 9 | Eurostat | PIB PPS + Población FUA | `Eurostat_extract.py` |
| 10 | OECD | Política ambiental (EPS) | `oecd_process.py` |
| 11 | InvestEU / EIB | Financiación verde EU | `investeu_process.py` |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.titlesize'] = 13
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

# Rutas relativas al directorio del proyecto
ROOT        = Path('.')
KUZNETS_CSV = ROOT / 'data' / 'DatosProcesados' / 'Kuznets.csv'
LORCA_PROC  = ROOT / 'Lorca' / 'ETL' / 'script' / 'DatosProcesados'
LORCA_DATA  = ROOT / 'Lorca' / 'data' / 'processed'

def try_load(paths, label):
    for p in paths:
        if Path(p).exists():
            df = pd.read_csv(p)
            print(f'[OK] {label} cargado desde {p}  →  {df.shape}')
            return df
    print(f'[--] {label} no disponible localmente (generado en Lorca)')
    return None

print(f'Dataset maestro : {KUZNETS_CSV}')
print(f'Existe          : {KUZNETS_CSV.exists()}')

---
## 1. Dataset Maestro — `Kuznets.csv`

Output de `merge.py`: fusión de todas las fuentes en un panel `ciudad × año × mes`.  
Es el input directo del pipeline de ingesta a HDFS Bronze en el cluster Lorca.

In [ ]:
df = pd.read_csv(KUZNETS_CSV, low_memory=False)

print(f'Shape      : {df.shape[0]:,} filas  ×  {df.shape[1]} columnas')
print(f'Memoria    : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Tipos de datos:')
print(df.dtypes.value_counts().to_string())

In [ ]:
# Cobertura geográfica y temporal
n_cities   = df['City'].nunique() if 'City' in df.columns else '—'
years      = sorted(df['Year'].unique()) if 'Year' in df.columns else []
n_countries = df['Country_Code'].nunique() if 'Country_Code' in df.columns else '—'
months     = sorted(df['Month'].unique()) if 'Month' in df.columns else []

print(f'Ciudades únicas    : {n_cities}')
print(f'Países cubiertos   : {n_countries}')
print(f'Años disponibles   : {years}')
print(f'Meses disponibles  : {months}')
print()
df.head(3)

---
## 2. Análisis Global de Valores Ausentes

El patrón de valores ausentes revela la **disponibilidad real de cada fuente** y
guía las decisiones de imputación en el pipeline de transformación.

In [ ]:
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'N ausentes': missing, '% ausentes': missing_pct})
miss_df = miss_df[miss_df['N ausentes'] > 0].sort_values('% ausentes', ascending=False)

if not miss_df.empty:
    fig, ax = plt.subplots(figsize=(12, max(4, len(miss_df) * 0.35)))
    miss_df['% ausentes'].plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_xlabel('% valores ausentes')
    ax.set_title('Valores ausentes por columna')
    plt.tight_layout()
    plt.show()
    print(miss_df.to_string())
else:
    print('Sin valores ausentes detectados en el dataset maestro.')

---
## 3. NDVI — Índice de Vegetación (Sentinel-2)

**Fuente:** GEE `COPERNICUS/S2_SR_HARMONIZED`  
**Cálculo:** `NDVI = (B8 − B4) / (B8 + B4)` · Resolución 10 m, buffer 20 km  
**Filtrado:** Máscara de nubes QA60 (bits 10–11), umbral nubosidad < 60 %  
**Rango físico:** [−1, 1] · Vegetación densa: > 0.5

In [ ]:
ndvi_cols = [c for c in df.columns if 'NDVI' in c.upper() and 'SLOPE' not in c.upper()]
print(f'Columnas NDVI encontradas: {ndvi_cols}')

if ndvi_cols:
    print('\nEstadísticas descriptivas:')
    print(df[ndvi_cols].describe().round(4).to_string())

    # Validación del rango físico
    if 'NDVI_Mean' in df.columns:
        fuera_rango = df[(df['NDVI_Mean'] < -1) | (df['NDVI_Mean'] > 1)]
        print(f'\nValores fuera de [-1, 1]: {len(fuera_rango)} ({len(fuera_rango)/len(df)*100:.3f}%)')
        print('→ Artefactos de cálculo GEE — filtrados en extracción antes de write')

In [ ]:
if 'NDVI_Mean' in df.columns and 'Year' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Distribución NDVI_Mean
    df['NDVI_Mean'].dropna().hist(bins=60, ax=axes[0], color='#27ae60', edgecolor='white')
    axes[0].axvline(df['NDVI_Mean'].median(), color='red', linestyle='--',
                    label=f"Mediana: {df['NDVI_Mean'].median():.3f}")
    axes[0].set_title('Distribución NDVI_Mean')
    axes[0].set_xlabel('NDVI')
    axes[0].legend()

    # Evolución temporal
    ndvi_anual = df.groupby('Year')['NDVI_Mean'].mean()
    ndvi_anual.plot(ax=axes[1], marker='o', color='#27ae60', linewidth=2)
    axes[1].set_title('Evolución NDVI medio — Europa')
    axes[1].set_xlabel('Año')
    axes[1].set_ylabel('NDVI')

    # NDVI por país (top 10)
    if 'Country_Code' in df.columns:
        ndvi_pais = df.groupby('Country_Code')['NDVI_Mean'].mean().sort_values(ascending=False).head(12)
        ndvi_pais.plot(kind='bar', ax=axes[2], color='#27ae60', edgecolor='white')
        axes[2].set_title('NDVI medio por país')
        axes[2].set_xlabel('')
        axes[2].tick_params(axis='x', rotation=45)

    plt.suptitle('NDVI — Sentinel-2', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

### Decisiones de limpieza — NDVI

| Problema | Causa | Decisión |
|----------|-------|----------|
| Píxeles nubosos | Refletancia de nube → NDVI artificialmente bajo | Máscara QA60 (bits 10–11) aplicada en GEE antes del reduceRegion |
| Mes sin dato | < 5 píxeles válidos en el mes | GEE devuelve `None` → la fila no se escribe en el CSV |
| Valores > 1 o < −1 | Artefactos de división por cero en bandas saturadas | Filtrado en GEE antes de exportar |
| Agregación mensual → anual | merge.py necesita granularidad anual | Media de meses con dato disponible (no relleno de meses faltantes) |

---
## 4. NO₂ Troposférico — Calidad del Aire (Sentinel-5P)

**Fuente:** GEE `COPERNICUS/S5P/OFFL/L3_NO2` (producto Offline — mayor calidad que NRTI)  
**Banda:** `tropospheric_NO2_column_number_density` (mol/m²)  
**Filtrado:** `cloud_fraction < 0.3` aplicado en GEE  
**Relación con EKC:** NO₂ actúa como variable ambiental alternativa a NDVI

In [ ]:
no2_cols = [c for c in df.columns if 'NO2' in c.upper()]
print(f'Columnas NO₂: {no2_cols}')

if no2_cols:
    print('\nEstadísticas:')
    print(df[no2_cols].describe().round(6).to_string())

    # Valores negativos: esperados en S5P (ruido de fondo)
    if 'NO2_Mean' in df.columns:
        neg = df[df['NO2_Mean'] < 0]
        print(f'\nValores NO₂ < 0: {len(neg)} ({len(neg) / len(df.dropna(subset=["NO2_Mean"])) * 100:.2f}%)')
        print('→ Físicamente posibles (noise floor del sensor) — se mantienen en el dataset')

In [ ]:
if 'NO2_Mean' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribución NO2
    df['NO2_Mean'].dropna().hist(bins=60, ax=axes[0], color='#8e44ad', edgecolor='white')
    axes[0].set_title('Distribución NO₂_Mean (mol/m²)')
    axes[0].set_xlabel('NO₂_Mean')

    # Scatter NDVI vs NO2
    if 'NDVI_Mean' in df.columns:
        valid = df[['NDVI_Mean', 'NO2_Mean']].dropna()
        r = valid.corr().iloc[0, 1]
        axes[1].scatter(valid['NO2_Mean'], valid['NDVI_Mean'],
                        alpha=0.07, s=4, color='#8e44ad')
        axes[1].set_xlabel('NO₂_Mean')
        axes[1].set_ylabel('NDVI_Mean')
        axes[1].set_title(f'NDVI vs NO₂  (r = {r:.3f})')
        axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)

    plt.suptitle('NO₂ — Sentinel-5P', fontsize=14)
    plt.tight_layout()
    plt.show()

    print('Una correlación negativa NDVI↑ NO₂↓ sería evidencia de EKC:')
    print('ciudades con más vegetación tienden a tener menos contaminación.')

---
## 5. Impermeabilización del Suelo — HRL Copernicus

**Fuente:** EEA High Resolution Layers (HRL) — Imperviousness Density + Tree Cover Density  
**Formato original:** GeoTIFF 10 m → convertido a CSV con `rasterio` en `hrl_to_csv.py`  
**Años disponibles:** 2018 y 2021 únicamente (ciclos trienales EEA)  
**Valor nodata:** 255 en el GeoTIFF original

In [ ]:
hrl_cols = [c for c in df.columns if any(k in c.lower() for k in ['imperv', 'tree', 'woody'])]
print(f'Columnas HRL: {hrl_cols}')

if hrl_cols:
    print('\nEstadísticas:')
    print(df[hrl_cols].describe().round(2).to_string())

    # Distribución
    n = min(len(hrl_cols), 3)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, hrl_cols[:3]):
        df[col].dropna().hist(bins=40, ax=ax, color='#e67e22', edgecolor='white')
        ax.set_title(col)
        ax.set_xlabel('%')
    plt.suptitle('HRL — Impermeabilización y cobertura vegetal', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Cobertura por año
    if 'Year' in df.columns:
        cobertura = df.groupby('Year')[hrl_cols[0]].count()
        print(f'\nRegistros no nulos de {hrl_cols[0]} por año:')
        print(cobertura.to_string())
        print('→ Solo 2018 y 2021 tienen datos reales; el resto son imputados')
else:
    print('Columnas HRL no presentes en este dataset.')

### Decisiones de limpieza — HRL

| Problema | Causa | Decisión |
|----------|-------|----------|
| `nodata = 255` | Valor de nodata en GeoTIFF | Filtrado en `hrl_to_csv.py` antes de calcular la media |
| Valores > 100 o < 0 | Artefactos de reproyección | Eliminados (porcentaje no puede superar 100) |
| Años sin dato (2019, 2020, 2022…) | HRL solo cubre 2018 y 2021 | Imputación por vecino más cercano: años ≤ 2019 → 2018, años ≥ 2020 → 2021 |
| Baja variabilidad interanual | Imperviousness cambia muy lentamente | La imputación conservadora es adecuada para esta variable |

---
## 6. Datos Financieros — Yahoo Finance

**Fuente:** `yfinance` — 43 tickers europeos de energía verde y utilities  
**Granularidad:** Mensual (2006–2025) con `auto_adjust=True`  
**Volatilidad mensual:** `σ = std(r_diarios) × √21`  
**Retorno mensual:** `r = (P_cierre / P_apertura) − 1`  
**Filtro calidad:** mínimo 15 días de cotización en el mes

In [ ]:
fin_cols = [c for c in df.columns if c.startswith('Fin')]
fin_num  = [c for c in fin_cols if df[c].dtype in ['float64', 'int64']]
print(f'Columnas financieras: {fin_cols}')

if fin_num:
    print('\nEstadísticas financieras numéricas:')
    print(df[fin_num].describe().round(4).to_string())

    # Distribuciones
    n = min(len(fin_num), 3)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, fin_num[:3]):
        df[col].dropna().hist(bins=40, ax=ax, color='#2980b9', edgecolor='white')
        ax.set_title(col)
    plt.suptitle('Distribución variables financieras', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Outliers P1–P99
    for col in fin_num[:2]:
        q01, q99 = df[col].quantile([0.01, 0.99])
        out = df[(df[col] < q01) | (df[col] > q99)]
        print(f'Outliers {col} (P1–P99): {len(out)} ({len(out)/len(df)*100:.2f}%)')
    print('→ Outliers financieros son eventos reales (COVID 2020, crisis energética 2022) — se mantienen')
else:
    print('Columnas financieras numéricas no disponibles.')

---
## 7. Nuevas Fuentes ETL — Pipeline Lorca 2026

Las siguientes fuentes se incorporaron en la segunda versión del pipeline.  
No están en el CSV legacy `Kuznets.csv`; se encuentran en el pipeline HDFS de Lorca.  
Se muestra el esquema esperado, estadísticas si el archivo está disponible localmente,
y la justificación de inclusión.

In [ ]:
# ── ERA5-Land: temperatura y precipitación ──────────────────────────────────
era5 = try_load(
    [LORCA_DATA / 'era5_climate.csv', LORCA_PROC / 'era5_climate.csv'],
    'ERA5-Land'
)

if era5 is not None:
    print(era5.describe().round(2).to_string())
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    era5['Temp_Annual_Mean_C'].dropna().hist(bins=40, ax=axes[0], color='#e74c3c', edgecolor='white')
    axes[0].set_title('ERA5 — Temperatura media anual (°C)')
    era5['Precip_Annual_Sum_m'].dropna().hist(bins=40, ax=axes[1], color='#3498db', edgecolor='white')
    axes[1].set_title('ERA5 — Precipitación anual acumulada (m)')
    plt.tight_layout()
    plt.show()
else:
    print()
    print('Esquema esperado:')
    print('  City (str) | Year (int) | Temp_Annual_Mean_C (float) | Precip_Annual_Sum_m (float)')
    print()
    print('Transformaciones:')
    print('  - Conversión de temperatura: K → °C  (temp_c = temp_k - 273.15)')
    print('  - Sin valores ausentes: ERA5 es un reanálisis de cobertura global continua')
    print('  - Rango de validación: Temp [-60, 60] °C | Precip [0, 10] m/año')
    print()
    print('Justificación de inclusión:')
    print('  El clima es un confundidor clave en el modelo EKC: ciudades nórdicas')
    print('  tienen vegetación distinta a las mediterráneas con el mismo PIB.')
    print('  Incluir temp y precip evita que el clustering agrupe ciudades')
    print('  climáticamente incomparables.')

In [ ]:
# ── ESA WorldCover: uso del suelo ────────────────────────────────────────────
wc = try_load(
    [LORCA_DATA / 'worldcover.csv', LORCA_PROC / 'worldcover.csv'],
    'ESA WorldCover'
)

if wc is not None:
    pct_cols = [c for c in wc.columns if 'pct' in c.lower() or 'Pct' in c]
    if pct_cols:
        print('Estadísticas clases de cobertura:')
        print(wc[pct_cols].describe().round(2).to_string())
        fig, ax = plt.subplots(figsize=(12, 4))
        wc[pct_cols].mean().sort_values().plot(kind='barh', ax=ax, color='#16a085')
        ax.set_title('WorldCover — % medio por clase de cobertura')
        ax.set_xlabel('%')
        plt.tight_layout()
        plt.show()
else:
    print()
    print('Esquema esperado:')
    print('  City | Year | WC_Tree_Pct | WC_Built_Pct | WC_Crop_Pct | WC_Natural_Pct')
    print()
    print('Años disponibles: 2020 y 2021 únicamente (limitación de ESA WorldCover)')
    print()
    print('Decisiones de limpieza:')
    print('  - La suma de clases puede ser < 100% por píxeles de agua permanente o nieve')
    print('  - Si la suma < 95%: normalización dividiendo por suma de clases presentes')
    print('  - NaN para años distintos de 2020/2021 (no hay datos disponibles)')
    print()

# ── S5P Aerosol UVAI ─────────────────────────────────────────────────────────
uvai = try_load(
    [LORCA_DATA / 's5p_aerosol.csv', LORCA_PROC / 's5p_aerosol.csv'],
    'S5P Aerosol UVAI'
)

if uvai is None:
    print('Esquema UVAI esperado:')
    print('  City | Year | UVAI_Annual_Mean (float, rango ~-2 a +5)')
    print('  UVAI > 0 → aerosoles absorbentes (humo, polvo sahariano)')
    print('  UVAI < 0 → aerosoles dispersores (sal marina, nubes)')

In [ ]:
# ── EDGAR CO2 ─────────────────────────────────────────────────────────────────
edgar = try_load(
    [LORCA_DATA / 'edgar_co2.csv', LORCA_PROC / 'edgar_co2.csv'],
    'EDGAR CO₂'
)

if edgar is not None:
    print(edgar.describe().round(0).to_string())
else:
    print('Esquema EDGAR esperado: country_code | Year | CO2_Country_kt')
    print('Granularidad: país (expandido a todas las ciudades de ese país)')
    print()

# ── OECD Indicadores ambientales ──────────────────────────────────────────────
oecd = try_load(
    [LORCA_PROC / 'oecd_indicators.csv', LORCA_DATA / 'oecd_indicators.csv'],
    'OECD'
)

if oecd is not None:
    print(oecd.describe().round(2).to_string())
    anos_oecd = sorted(oecd['Year'].unique()) if 'Year' in oecd.columns else []
    print(f'Años OECD disponibles: {anos_oecd}')
else:
    print('Esquema OECD esperado: City | Year | EPS_Index | Env_Tax_USD | Env_Expenditure | GHG_Total_kt')
    print('EPS_Index: 0 = sin política ambiental | 6 = máxima rigurosidad')
    print('Decisiones: air_ghg.csv (múltiples gases) → suma todos por país-año → GHG_Total_kt')
    print('            env_policy (sub-indicadores) → media por país-año → EPS_Index')
    print()

# ── InvestEU ──────────────────────────────────────────────────────────────────
investeu = try_load(
    [LORCA_PROC / 'investeu_summary.csv', LORCA_DATA / 'investeu_summary.csv'],
    'InvestEU'
)

if investeu is not None:
    print(investeu.describe().round(0).to_string())
else:
    print('Esquema InvestEU esperado: City | Year | InvestEU_Ops_Count | InvestEU_Total_EUR')
    print('Cobertura: 2021+ (lanzamiento del programa InvestEU)')
    print('Año extraído de pdf_url (regex año 4 dígitos); fallback: extraction_date; default: 2022')
    print('~30-40% de registros raw no tienen año en la URL → default aplicado')

---
## 8. Análisis de Correlaciones

La correlación entre las variables ambientales y económicas es fundamental para
validar la hipótesis EKC: se espera **β₁ > 0, β₂ < 0** en la regresión de panel,
lo que implica una relación cuadrática entre GDP y degradación ambiental.

In [ ]:
exclude = {'Year', 'Month'}
num_cols = [
    c for c in df.columns
    if df[c].dtype in ['float64', 'int64']
    and c not in exclude
    and 'Ticker' not in c
    and 'Compan' not in c
]

if len(num_cols) >= 3:
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    fig, ax = plt.subplots(figsize=(min(16, len(num_cols) * 1.2), min(14, len(num_cols) * 1.0)))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, vmin=-1, vmax=1, ax=ax,
                annot_kws={'size': 8}, linewidths=0.4,
                square=True)
    ax.set_title('Matriz de correlación — variables cuantitativas', pad=15)
    plt.tight_layout()
    plt.show()

    # Top correlaciones con NDVI_Mean
    if 'NDVI_Mean' in corr.columns:
        top = corr['NDVI_Mean'].drop('NDVI_Mean').abs().sort_values(ascending=False).head(6)
        print('Top correlaciones con NDVI_Mean (|r|):')
        print(top.round(3).to_string())
else:
    print('Pocas columnas numéricas — ampliar el dataset con las nuevas fuentes.')

---
## 9. NDVI Slope — Señal del Turning Point

`NDVI_Slope` es la **pendiente de la regresión lineal OLS** de NDVI sobre el tiempo
para cada ciudad: `NDVI ~ β·año + α`.  
Una pendiente positiva indica **mejora ambiental activa** → señal principal del Turning Point.

In [ ]:
slope_col = next((c for c in df.columns if 'slope' in c.lower()), None)

if slope_col and 'City' in df.columns:
    city_slope = df.groupby('City')[slope_col].mean().dropna()

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Distribución de pendientes
    city_slope.hist(bins=40, ax=axes[0], color='#1abc9c', edgecolor='white')
    axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='slope = 0')
    axes[0].set_title(f'Distribución {slope_col} por ciudad')
    axes[0].set_xlabel('Pendiente NDVI/año')
    axes[0].legend()

    # Top 12 ciudades con mayor mejora
    top12 = city_slope.sort_values(ascending=False).head(12)
    top12.plot(kind='barh', ax=axes[1], color='#27ae60', edgecolor='white')
    axes[1].set_title('Top 12 ciudades — mayor mejora NDVI')
    axes[1].set_xlabel('Pendiente')

    plt.suptitle('NDVI Slope — Tendencia temporal por ciudad', fontsize=13)
    plt.tight_layout()
    plt.show()

    mejora    = (city_slope > 0).sum()
    degrada   = (city_slope < 0).sum()
    print(f'Ciudades con NDVI en mejora     : {mejora} ({mejora / len(city_slope) * 100:.1f}%)')
    print(f'Ciudades con NDVI en degradación: {degrada} ({degrada / len(city_slope) * 100:.1f}%)')
    print()
    print('Decisión: la pendiente se calcula en merge.py con scipy.stats.linregress')
    print('sobre los valores anuales de NDVI_Mean por ciudad (no mensuales).')
    print('Mínimo 3 años con dato para calcular la pendiente; de lo contrario NaN.')
else:
    print('NDVI_Slope no encontrado en este dataset.')

---
## 10. Cobertura y Calidad por Fuente

Resumen de la completitud de cada fuente en el dataset maestro.

In [ ]:
fuente_col_map = {
    'Sentinel-2 NDVI'    : [c for c in df.columns if 'NDVI' in c.upper() and 'SLOPE' not in c.upper()],
    'Sentinel-5P NO₂'    : [c for c in df.columns if 'NO2' in c.upper()],
    'HRL Imperviousness' : [c for c in df.columns if any(k in c.lower() for k in ['imperv', 'tree', 'woody'])],
    'Yahoo Finance'      : [c for c in df.columns if c.startswith('Fin') and df[c].dtype != 'object'],
    'NDVI Slope (ETL)'   : [c for c in df.columns if 'slope' in c.lower()],
}

rows = []
for fuente, cols in fuente_col_map.items():
    if not cols:
        rows.append({'Fuente': fuente, 'Columnas': 0, 'Cobertura (%)': 'N/A'})
        continue
    cobertura = (1 - df[cols].isnull().any(axis=1).mean()) * 100
    rows.append({'Fuente': fuente, 'Columnas': len(cols), 'Cobertura (%)': f'{cobertura:.1f}'})

# Fuentes nuevas (Lorca) — disponibles en Bronze/Silver HDFS
for fuente in ['ERA5-Land', 'S5P Aerosol UVAI', 'ESA WorldCover', 'EDGAR CO₂', 'OECD', 'InvestEU', 'Eurostat GDP']:
    rows.append({'Fuente': fuente, 'Columnas': '—', 'Cobertura (%)': 'En Bronze HDFS (Lorca)'})

cov_df = pd.DataFrame(rows)
cov_df

---
## 11. Resumen de Decisiones de Limpieza ETL

Tabla consolidada de todas las transformaciones aplicadas en el pipeline ETL,
con justificación empírica basada en el EDA anterior.

In [ ]:
decisiones = pd.DataFrame([
    ('S2 NDVI',        'Píxeles nubosos',              'Máscara QA60 bits 10–11 en GEE antes del reduceRegion'),
    ('S2 NDVI',        'Meses sin píxeles válidos',     'Si < 5 px válidos, GEE devuelve None → fila no escrita'),
    ('S2 NDVI',        'Valores fuera de [−1, 1]',      'Filtrado en GEE; artefactos de división por cero en bandas'),
    ('S2 NDVI',        'Agregación mensual → anual',    'Media de meses con dato; no se imputan meses faltantes'),
    ('S5P NO₂',        'Fracción nubosa',               'cloud_fraction < 0.3 en GEE antes de reduceRegion'),
    ('S5P NO₂',        'Valores negativos',             'Mantenidos: físicamente posibles (noise floor del sensor)'),
    ('S5P NO₂',        'Producto Offline vs NRTI',      'Se usa OFFL (procesamiento tardío): mayor precisión de retrieval'),
    ('HRL',            'nodata = 255 en GeoTIFF',       'Filtrado en hrl_to_csv.py antes de calcular media espacial'),
    ('HRL',            'Valores > 100 o < 0',           'Eliminados: son artefactos de reproyección del GeoTIFF'),
    ('HRL',            'Años sin datos (2019, 2020…)',   'Imputación vecino más cercano: ≤2019 → 2018; ≥2020 → 2021'),
    ('ERA5',           'Temperatura en Kelvin',         'Conversión K → °C en GEE: temp_c = temp_k - 273.15'),
    ('ERA5',           'Valores ausentes',              'Sin ausencias: ERA5 es reanálisis de cobertura global'),
    ('WorldCover',     'Años limitados',                'Solo 2020 y 2021; NaN para el resto de años del panel'),
    ('WorldCover',     'Suma clases ≠ 100%',            'Normalizar si suma < 95%; píxeles de agua/nieve excluidos'),
    ('EDGAR CO₂',      'Granularidad país',             'Expansión país → ciudades del mismo país (mismo valor)'),
    ('EDGAR CO₂',      'Mapeo ISO3 → ISO2',             'Diccionario ISO3_TO_ISO2 para 20+ países europeos en config'),
    ('Finance',        'Meses con < 15 días',           'Descartados: no representativos estadísticamente'),
    ('Finance',        'Splits y dividendos',           'auto_adjust=True en yfinance; series de precios homogéneas'),
    ('Finance',        'Outliers de volatilidad',       'Mantenidos: eventos reales (COVID 2020, crisis energética 2022)'),
    ('Finance',        'Agrupación empresa → país',     'Tickers agregados por sector-país para cruzar con GDP ciudad'),
    ('OECD env_policy','Sub-indicadores por gas/sector','Media por país-año → EPS_Index único por fila'),
    ('OECD air_ghg',   'Múltiples gases GHG',           'Suma de todos los gases por país-año → GHG_Total_kt (CO₂eq)'),
    ('OECD/InvestEU',  'Granularidad país',             'Expansión país → ciudades (mismo valor para todas en el país)'),
    ('InvestEU',       'Año no en URL del PDF',         'Fallback: extraction_date; si tampoco → default 2022'),
    ('Merge final',    'Base del left join',            'Sentinel-2 es la fuente más completa → ancla del panel'),
    ('Merge final',    'Duplicados city×year',          'groupby(City, Year).mean() antes de join en merge.py'),
], columns=['Fuente', 'Problema detectado', 'Transformación aplicada'])

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.max_rows', 30)
decisiones

---
## 12. Conclusiones

### Variables con mayor relevancia para el modelo EKC

| Variable | Rol | Completitud |
|----------|-----|-------------|
| `NDVI_Mean` | Variable dependiente ambiental (ln E) | Alta (S2 post-cloud mask) |
| `NO2_Mean` | Variable dependiente alternativa | Alta |
| `gdp_pps_per_capita` | Variable explicativa (ln Y, ln Y²) | Alta (Eurostat) |
| `NDVI_Slope` | Target XGBoost / señal Turning Point | Derivada del panel |
| `Imperviousness_Mean` | Variable de presión urbana | Media (solo 2018/2021) |
| `EPS_Index` | Variable de política ambiental | Media (~20 países) |
| `investeu_total_eur` | Proxy inversión verde | Baja (2021+) |

### Limitaciones identificadas en el EDA

1. **HRL y WorldCover** tienen muy pocos años disponibles — las imputaciones introducen sesgo para análisis de tendencia de uso del suelo.
2. **OECD e InvestEU** tienen cobertura temporal y de países incompleta — no todos los país-años tienen dato.
3. **Finance** agrega tickers a nivel sector-país, no ciudad — pierde variabilidad intraterritorial.
4. **WorldCover** solo cubre 2020-2021 — no permite análisis de tendencia de cobertura del suelo.

### Arquitectura de transformación

El pipeline ETL implementado en Lorca (Bronze → Silver → Gold) garantiza:
- **Bronze:** datos brutos inmutables con metadata de auditoría (`_ingestion_date`, `_source_system`)
- **Silver:** datos limpios en Star Schema Kimball con SOURCE_CATALOG de 12 entradas
- **Gold:** métricas calculadas y resultados ML listos para el cuadro de mando